In [ ]:
import os
import json
import requests
from langchain_openai import ChatOpenAI

# 1. Config
MCP_SERVER_URL = "http://localhost:8000/mcp"
os.environ["OPENAI_API_KEY"]  # make sure your API key is set in env

# 2. Setup LLM
llm = ChatOpenAI(model="gpt-4o-mini")


In [2]:

# 3. Define helper to call MCP tools
def call_mcp_tool(tool_name: str, payload: dict):
    """Call MCP tool by name with JSON payload"""
    url = f"{MCP_SERVER_URL}/tools/{tool_name}"
    resp = requests.post(url, json=payload)
    return resp.json()


In [22]:
import asyncio
from langgraph.prebuilt import create_react_agent
from langchain_mcp_adapters.client import MultiServerMCPClient

async def get_mcp_list():
    """Call MCP tool by name with JSON payload"""
    servers = {
        "orian": {
            "url": "http://localhost:8000/mcp",
            "transport": "streamable_http"
        }
    }

    client = MultiServerMCPClient(servers)
    tools = await client.get_tools()   # this internally calls session/start → tools/list

    return tools

# 4. Example usage
tools = await get_mcp_list()
print("Available tools:", tools)

Available tools: [StructuredTool(name='get_orian_status', description='\n    Mirrors GET /admin/get_status as an MCP tool.\n    ', args_schema={'properties': {}, 'title': 'get_orian_statusArguments', 'type': 'object'}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x00000163638C0160>), StructuredTool(name='get_weather', description='\n    Mirrors GET /admin/get_weather as an MCP tool.\n    ', args_schema={'properties': {}, 'title': 'get_weatherArguments', 'type': 'object'}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x00000163638C2B00>)]


C:\Users\madhu\AppData\Local\Temp\ipykernel_23724\1596288425.py:20: RuntimeWarning: coroutine 'get_mcp_list' was never awaited
  tools = await get_mcp_list()


In [ ]:
from langchain_openai import ChatOpenAI
from langchain.agents import initialize_agent, AgentType

# LLM
llm = ChatOpenAI(model="gpt-4o-mini")

# Suppose you already have your tools as `mcp_tools = [tool1, tool2]`
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.OPENAI_MULTI_FUNCTIONS,  # lets LLM decide which tool
    verbose=True
)

query = "You are a helpful assistant that can use various tools. Please help me with the following task: Find the current weather in New York City and get the orian status."

res = await agent.ainvoke({"input": query})
print(res)




> Entering new AgentExecutor chain...

Invoking: `get_weather` with `{'location': 'New York City'}`



Invoking: `get_orian_status` with `{}`


{
  "status": "running"
}{
  "weather": "sunny"
}The current weather in New York City is sunny, and the Orian status is running.

> Finished chain.
{'input': 'You are a helpful assistant that can use various tools. Please help me with the following task: Find the current weather in New York City and get the orian status.', 'output': 'The current weather in New York City is sunny, and the Orian status is running.'}
